# Starbucks Competitor Analysis in Los Angeles

Which Starbucks locations in Los Angeles have the **fewest competitor coffee shops and cafes** within a 10-minute driving isochrone? And which zip codes contain these underserved locations?

## Data sources

| Dataset | Table | Description |
|---|---|---|
| Overture Places + Isochrones | `wherobots_pro_data.overture_maps_foundation.overture_places_with_isochrones` | POI data with pre-computed 5/10/15/20-min driving isochrones |
| US Census ZCTA | `wherobots_pro_data.us_census.zipcode` | ZIP Code Tabulation Area boundaries |

## Approach

1. Extract all Starbucks locations in LA with their 10-minute driving isochrones
2. Extract all competitor coffee shops and cafes in LA (excluding Starbucks)
3. Spatial join: count competitors within each Starbucks' 10-min isochrone
4. Cross-reference with ZCTA boundaries to identify underserved zip codes

## 1. Set up the Sedona context

In [ ]:
from sedona.spark import *

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## 2. Starbucks locations in Los Angeles

Query all Starbucks in the city of Los Angeles that have a 10-minute driving isochrone available.

In [ ]:
starbucks = sedona.sql("""
SELECT id, names.primary AS name, geometry,
       addresses[0].freeform AS address,
       addresses[0].postcode AS postcode,
       isochrone_10min
FROM wherobots_pro_data.overture_maps_foundation.overture_places_with_isochrones
WHERE brand.names.primary = 'Starbucks'
  AND addresses[0].region = 'CA'
  AND addresses[0].locality = 'Los Angeles'
  AND isochrone_10min IS NOT NULL
""")

starbucks.createOrReplaceTempView('starbucks')
print(f"Starbucks locations in LA: {starbucks.count()}")
starbucks.show(5, truncate=False)

## 3. Competitor coffee shops and cafes

Find all coffee shops and cafes in Los Angeles that are **not** Starbucks. We filter on both `brand.names.primary` and `names.primary` to catch locations where the brand field is null.

In [ ]:
competitors = sedona.sql("""
SELECT id, names.primary AS name, geometry,
       categories.primary AS category,
       brand.names.primary AS brand_name
FROM wherobots_pro_data.overture_maps_foundation.overture_places_with_isochrones
WHERE categories.primary IN ('coffee_shop', 'cafe')
  AND addresses[0].region = 'CA'
  AND addresses[0].locality = 'Los Angeles'
  AND names.primary != 'Starbucks'
  AND (brand.names.primary != 'Starbucks' OR brand.names.primary IS NULL)
""")

competitors.createOrReplaceTempView('competitors')
print(f"Competitor coffee shops/cafes in LA: {competitors.count()}")
competitors.show(10, truncate=False)

## 4. Count competitors within each Starbucks' 10-minute driving isochrone

For each Starbucks, use `ST_Contains` to find how many competitor locations fall inside its pre-computed 10-minute driving isochrone polygon.

In [ ]:
starbucks_competitors = sedona.sql("""
SELECT s.id, s.name, s.address, s.postcode, s.geometry,
       COUNT(c.id) AS competitor_count
FROM starbucks s
LEFT JOIN competitors c
  ON ST_Contains(s.isochrone_10min, c.geometry)
GROUP BY s.id, s.name, s.address, s.postcode, s.geometry
ORDER BY competitor_count ASC
""")

starbucks_competitors.createOrReplaceTempView('starbucks_competitors')
starbucks_competitors.show(20, truncate=False)

## 5. Cross-reference with ZCTA boundaries

Join each Starbucks location to its containing ZIP Code Tabulation Area, then aggregate to see which zip codes have the least competition on average.

In [ ]:
zip_competition = sedona.sql("""
SELECT z.ZCTA5CE10 AS zip_code,
       z.geometry AS zip_geometry,
       COUNT(sc.id) AS starbucks_count,
       ROUND(AVG(sc.competitor_count), 1) AS avg_competitors,
       MIN(sc.competitor_count) AS min_competitors,
       MAX(sc.competitor_count) AS max_competitors
FROM starbucks_competitors sc
JOIN wherobots_pro_data.us_census.zipcode z
  ON ST_Contains(z.geometry, sc.geometry)
GROUP BY z.ZCTA5CE10, z.geometry
ORDER BY avg_competitors ASC
""")

zip_competition.createOrReplaceTempView('zip_competition')
print(f"Zip codes with Starbucks locations: {zip_competition.count()}")
zip_competition.show(30, truncate=False)

## 6. Visualize underserved Starbucks locations

Map each Starbucks location, colored by how many competitors are within its 10-minute drive time. Lower competitor counts indicate underserved locations.

In [ ]:
map_locations = SedonaKepler.create_map()
SedonaKepler.add_df(map_locations, starbucks_competitors, 'Starbucks Competitor Count')
map_locations

## 7. Visualize zip codes by average competitor density

Map ZCTA boundaries colored by average competitor count to identify underserved zip codes at a glance.

In [ ]:
map_zips = SedonaKepler.create_map()
SedonaKepler.add_df(map_zips, zip_competition, 'Zip Code Competition')
map_zips

## 8. Detailed view: most underserved Starbucks locations

The individual Starbucks locations with the fewest competitors within a 10-minute drive, along with their zip code.

In [ ]:
underserved = sedona.sql("""
SELECT sc.name, sc.address, sc.postcode, sc.competitor_count,
       z.ZCTA5CE10 AS zip_code
FROM starbucks_competitors sc
JOIN wherobots_pro_data.us_census.zipcode z
  ON ST_Contains(z.geometry, sc.geometry)
ORDER BY sc.competitor_count ASC
""")

underserved.show(30, truncate=False)